# Solutions — Reducers

Only look here after you've actually tried the exercises in `usereducer.ipynb`.

### LESSON 52 — Exercise

**Part 1.**

In [ ]:
const l52initial = { tasks: [], filter: "all" };

function l52reducer(state, action) {
  switch (action.type) {
    case "task_added":
      return { ...state, tasks: [...state.tasks, { id: action.id, text: action.text, done: false }] };

    case "task_toggled":
      return {
        ...state,
        tasks: state.tasks.map((t) => (t.id === action.id ? { ...t, done: !t.done } : t)),
      };

    case "task_removed":
      return { ...state, tasks: state.tasks.filter((t) => t.id !== action.id) };

    case "filter_changed":
      return { ...state, filter: action.filter };

    // 1 - edit one task's text
    case "task_edited":
      return {
        ...state,
        tasks: state.tasks.map((t) => (t.id === action.id ? { ...t, text: action.text } : t)),
      };

    // 2 - drop everything already done
    case "completed_cleared":
      return { ...state, tasks: state.tasks.filter((t) => !t.done) };

    // 3 - decided FROM the state, which is what a reducer is for
    case "all_toggled": {
      const everyoneDone = state.tasks.length > 0 && state.tasks.every((t) => t.done);
      return { ...state, tasks: state.tasks.map((t) => ({ ...t, done: !everyoneDone })) };
    }

    default:
      throw new Error(`Unknown action: ${action.type}`);
  }
}

// A starting state with something in it
const l52start = l52reducer(
  l52reducer(l52initial, { type: "task_added", id: "a", text: "Write" }),
  { type: "task_added", id: "b", text: "Review" },
);

// 4 - purity: same input twice, same output; and the input is untouched
const l52snapshot = JSON.stringify(l52start);

for (const action of [
  { type: "task_edited", id: "a", text: "Write it properly" },
  { type: "completed_cleared" },
  { type: "all_toggled" },
]) {
  const once = l52reducer(l52start, action);
  const twice = l52reducer(l52start, action);
  console.log(
    action.type.padEnd(18),
    "deterministic:", JSON.stringify(once) === JSON.stringify(twice),
    "| input untouched:", JSON.stringify(l52start) === l52snapshot,
    "| new object:", !Object.is(once, l52start),
  );
}

console.log("");
console.log("all_toggled twice returns to the start?",
  JSON.stringify(l52reducer(l52reducer(l52start, { type: "all_toggled" }), { type: "all_toggled" })) === l52snapshot);

That last check is a small gift from purity: because `all_toggled` decides from the state and
never mutates, applying it twice lands exactly where you began. You can only assert that about
a function with no memory of its own.

Note `all_toggled` guards `state.tasks.length > 0`. Without it, `every` on an empty array is
`true`, so toggling an empty list would mark nothing as "undone" — harmless here, and exactly
the kind of edge case a reducer makes easy to see because it is all in one place.

**Part 2 — actions that describe versus actions that instruct.**

In [ ]:
// DESCRIBES what happened:
//   { type: "task_added", id: "x", text: "Hello" }
//   { type: "filter_changed", filter: "done" }
//   { type: "task_removed", id: "x" }
//
// INSTRUCTS the state (rewrite these):
//   { type: "set_tasks", tasks: [] }
//        -> what actually happened? Probably "all_cleared" or "tasks_loaded".
//           "set_tasks" tells you nothing about why the list is now empty.
//   { type: "set_filter", filter: "done" }
//        -> "filter_changed". Same payload, and now the name says a user did something.
//   { type: "increment_count_by", amount: 1 }
//        -> this one is subtler. It describes a CHANGE, not an event. What happened was
//           something like "item_added" or "vote_cast"; the fact that the count goes up by
//           one is the reducer's business, not the caller's.
//
// The test: could you read the action log out loud as a story of what the user did? If an
// entry sounds like a line of code rather than a sentence, it is an instruction.

console.log("actions are events, not assignments");

**Common mistakes.**

- Mutating and returning the same object. It "works" in a notebook and breaks every comparison
  React makes.
- Putting `Math.random()` or `Date.now()` in a reducer. Generate the id where the action is
  created and pass it in — which is why `task_added` carries an `id`.
- Forgetting `default`, so a typo does nothing at all.
- Naming everything `set_*`. That is `useState` with more typing.

### LESSON 52 — Mini challenge

**A — it mutates.** `state.tasks.push(...)` edits the array it was handed, and then returns the
**same** state object. Two failures for the price of one: the previous state is now corrupted
(so nothing can compare before and after), and the returned state is `Object.is`-equal to the
old one, so React would skip the re-render entirely. In a real app: the task is added and the
screen never shows it, until something unrelated re-renders and it appears from nowhere.

**B — it is not pure.** `crypto.randomUUID()` means the same `(state, action)` produces a
different result every call. The reducer can no longer be tested by comparing outputs, replayed
to reconstruct state, or called twice safely — and React's Strict Mode does call it twice in
development, so you would get two different ids for one action. The id belongs in the action:
`{ type: "task_added", id: crypto.randomUUID(), text }`, created by the caller.

**C — no `default`.** A `switch` with no matching case and no default returns `undefined`, so
one typo replaces your entire state with nothing. Every read downstream then throws, far from
the cause. Either throw (loud, best while developing) or `return state` (safe), but never fall
off the end.

**D — nothing is wrong with it.**

**Why D is fine and `set_tasks` is not.** They look identical in shape — an action carrying a
whole array that replaces the list. The difference is what they *name*.

`tasks_loaded` describes a real event: the data arrived from somewhere. That genuinely happened,
it is the kind of thing you would write in a log, and the reducer is the right place to decide
what the state looks like afterwards.

`set_tasks` describes nothing. It says only "assign this", which means the decision about what
the tasks should be was made somewhere else — in a component, probably, where it cannot be
tested or reused.

So the rule is not "never send an array" or "never replace state". It is: **the action's name
must be an event, not an assignment.** The payload can be as big as it needs to be.

### LESSON 53 — Exercise

In [ ]:
// 1. Adding a task logs:
//
//      dispatch — task_added "Write the brief"
//      reducer — task_added
//      reducer — task_added
//      render — 1 tasks, filter "all"
//      render — 1 tasks, filter "all"
//
//    One dispatch because you sent one action. TWO reducer calls because Strict Mode calls
//    the reducer twice in development to check it is pure - "The result from one of the calls
//    is ignored." The render lines are doubled for the same reason (LESSON 3). In production
//    there is one of each.
//
// 2. Logging state.tasks.length straight after dispatch prints 0 for the first task.
//    LESSON 26: dispatch, like a setter, "only updates the state variable for the next
//    render". The `state` variable in the running handler belongs to the render that created
//    it and cannot change. React's own example says the same: "Still 42!"
//
// 3. Moving crypto.randomUUID() into the reducer makes it impure. Strict Mode calls it twice,
//    each call invents a DIFFERENT id, and the call React keeps is not necessarily the one
//    whose id you would guess. In the Components tab you see task ids that change between
//    renders, and keys that are not stable (LESSON 20) - React can no longer match a row to
//    the same task across renders, so toggling or removing can hit the wrong row.
//    Strict Mode makes it obvious precisely because the two calls disagree; without it the
//    bug would be invisible until something re-rendered.
//
// 4. A "cleared" action is one case and one button:
//
//      case "cleared":
//        return { ...state, tasks: [] };
//
//    The filter buttons keep working because the filter lives in the same state object and
//    was not touched - the spread carried it across.

console.log("one dispatch, two reducer calls, one kept");

**Common mistakes.**

- Reading `state` after `dispatch` and believing the dispatch failed.
- Generating ids, timestamps or random values inside the reducer.
- Replacing every `useState` in a component with one reducer. The experiment deliberately keeps
  `useState` for the text being typed; mixing them is normal.
- Treating the doubled reducer call as a bug to suppress rather than a purity check.

### LESSON 53 — Mini challenge

| | | why |
|---|---|---|
| 1. dark-mode checkbox | **useState** | one independent boolean |
| 2. multi-step wizard | **useReducer** | step, answers and errors change together per event |
| 3. search box text | **useState** | one controlled value (LESSON 30) |
| 4. shopping basket | **useReducer** (+ derived total) | items, quantities and discount move together |
| 5. dropdown open | **useState** | one independent boolean |
| 6. canvas with undo | **useReducer** | see below |

**The computed total.** LESSON 29 — and LESSON 44 said it again for Effects. A total is derived
from the items, so it does not belong in the reducer's state any more than it belonged in
`useState`. Compute it during render from `state.items`. Putting it in the reducer means every
action that touches items must remember to recompute it, which is the same drift, in a new
place.

**Why undo genuinely needs a reducer.** Because every change goes through **one function that
takes the previous state and returns the next**, you can keep a list of past states, or a list
of the actions themselves, and move backwards through it. With scattered setters there is no
single point where "a change happened" exists, so there is nothing to record.

The property doing the work is **purity**: the same action applied to the same state always
gives the same result, so replaying a history of actions reconstructs a state exactly. That is
also why `l54replay` in the next lesson is three lines.

### LESSON 54 — Exercise

**Part 1.**

In [ ]:
function l54createInitialState(savedText) {
  console.log("   createInitialState ran");
  const tasks = savedText.split(",").map((p) => p.trim()).filter(Boolean)
    .map((text, i) => ({ id: `s${i}`, text, done: false }));
  return { tasks, filter: "all", draft: "", error: null };
}

function l54reducer(state, action) {
  switch (action.type) {
    case "task_submitted": {
      if (action.text.trim() === "") return { ...state, error: "A task needs some text." };
      return {
        ...state,
        tasks: [...state.tasks, { id: action.id, text: action.text.trim(), done: false }],
        draft: "",
        error: null,
      };
    }

    case "draft_changed":
      return { ...state, draft: action.text };

    case "filter_changed":
      return { ...state, filter: action.filter };

    // 1 - clears the error and nothing else
    case "error_dismissed":
      return { ...state, error: null };

    // 2 - adds a task AND makes sure it can be seen.
    //     "visible" here means: if the filter is "done", a brand-new (undone) task would be
    //     hidden, so switch to "all". If the filter already shows undone tasks, leave it -
    //     changing a filter the user chose is worse than the problem it solves.
    case "task_submitted_visibly": {
      const next = l54reducer(state, { type: "task_submitted", id: action.id, text: action.text });
      if (next.error) return next;
      return { ...next, filter: next.filter === "done" ? "all" : next.filter };
    }

    default:
      throw new Error(`Unknown action: ${action.type}`);
  }
}

// 3 - replay: this is Array.prototype.reduce with the reducer as the callback
function l54replay(initialState, actions) {
  return actions.reduce(l54reducer, initialState);
}

const l54start = l54createInitialState("Write the brief, Book the room");

const l54final = l54replay(l54start, [
  { type: "draft_changed", text: "Send invites" },
  { type: "task_submitted", id: "n1", text: "Send invites" },
  { type: "filter_changed", filter: "done" },
  { type: "task_submitted_visibly", id: "n2", text: "Print badges" },
  { type: "error_dismissed" },
]);

console.log("tasks :", l54final.tasks.map((t) => t.text).join(" | "));
console.log("filter:", l54final.filter, "<- switched back from 'done' so the new task shows");
console.log("draft :", JSON.stringify(l54final.draft), "| error:", l54final.error);
console.log("start untouched?", l54start.tasks.length === 2);

// 4 - the initialiser logged once above because we called it once ourselves. Written as
//     useReducer(reducer, savedText, l54createInitialState) React also calls it once.
//     Written as useReducer(reducer, l54createInitialState(savedText)) it would run on EVERY
//     render - splitting the string, mapping it, building objects - and throw the result away
//     every time after the first.

Note that `task_submitted_visibly` **calls the reducer** rather than duplicating its logic.
A reducer is a plain function, so composing one case out of another is ordinary JavaScript —
and it keeps the validation rule in exactly one place.

**Part 2 — the playground, honestly assessed.**

Moving `draft` into the reducer gains: one source of truth for the form, a single
`task_submitted` action that clears the draft and the error together, and a transition you can
test without rendering.

It costs: every keystroke now dispatches an action and runs the whole reducer, which is more
machinery than `setText` for a value nobody else needs; the reducer grows a case that is pure
plumbing; and the component is slightly harder to skim, because reading "what happens when I
type" now means opening another function.

An honest verdict: for this experiment, **the split version is fine and arguably better**.
`draft` is independent, changes on its own, and nothing else depends on it — LESSON 53's own
rule says keep that in `useState`. It becomes worth moving the moment a second thing depends on
the draft, such as a live validation message or an unsaved-changes warning.

### LESSON 54 — Mini challenge

**The five problems.**

1. **`setUser` is an instruction, not an event** (LESSON 52). What happened was a sign-in:
   `user_signed_in`.
2. **`Date.now()` in the reducer** — impure. The same `(state, action)` gives a different result
   every call, and Strict Mode's two calls produce two different timestamps. It belongs in the
   action.
3. **`addItem` mutates.** `state.items.push(...)` edits the previous state, and the `{ ...state }`
   spread afterwards does not undo it — it makes a new outer object around the *same*, already
   damaged array (LESSON 28's shallow-copy trap).
4. **`updateTotal` stores a derived value** (LESSON 29). The total is a function of the items;
   keeping it in state means every item change must remember to dispatch `updateTotal`, and one
   day it will not.
5. **No `default` case** (LESSON 52). An unknown type returns `undefined` and wipes the state.
   Also, `setLoadingTrue` / `setLoadingFalse` should be one action with the reason for the
   change — or, better, two events like `fetch_started` and `fetch_succeeded`.

**The same problem twice: 1 and 5's loading pair.** `setUser`, `setLoadingTrue` and
`setLoadingFalse` are all assignments dressed as actions. Each names a field and a value rather
than an occurrence, which pushes the decision about *when* to set it back into the component —
exactly what the reducer was supposed to collect.

**The case that should not exist: `updateTotal`.** Renaming it `total_recalculated` would make
it sound like an event and change nothing. The total is derived, so no action should ever
produce it; it is computed during render from `state.items`. LESSON 29 said it first, LESSON 44
said it for Effects, and it is true a third time here.